# Event display examples

This notebook builds `track_vis.EventDisplay` scenes, first for a synthetic event and then for a real MC event read straight from a ROOT ntuple. A scene is drawn in the browser with three.js and has these layers:

- `add_tracks`: per-particle 3D trajectories, colour-coded by time, one toggleable layer per species
- `add_vertices`: MC truth vertex markers, grouped by species
- `add_deposits`: deposited energy binned into voxels and ray-marched on the GPU (the same renderer as `add_volume`, which takes a reconstructed image)

The track and vertex methods take columnar arrays that match the SNO+ `imaging_reco` ntuple schema. Tracks have a flat `pdg` array plus jagged (ragged) `pos_x`/`pos_y`/`pos_z`/`time`/`deposited_energy` arrays, with outer index = track and inner index = step.

A scene renders itself when it is the last expression in a cell. Drag to rotate, scroll to zoom, right-drag to pan. Hover over a track step or vertex to read its values.


In [ ]:
import awkward as ak
import numpy as np

from track_vis import EventDisplay, load_event

rng = np.random.default_rng(42)

## Build synthetic tracks

Each track is a helical/curved random walk from a common vertex. `pdg` is a flat list (one entry per track); `pos_x`, `pos_y`, `pos_z`, `time`, `deposited_energy` are jagged, one list of steps per track.

In [ ]:
def make_track(start, direction, n_steps, step_len, t0, dedx, curl=0.0):
    """Fake a particle trajectory: a straight-ish path that curls by `curl` rad/step."""
    direction = direction / np.linalg.norm(direction)
    perp = np.cross(direction, [0, 0, 1])
    if np.linalg.norm(perp) < 1e-6:
        perp = np.cross(direction, [0, 1, 0])
    perp = perp / np.linalg.norm(perp)

    positions = [start]
    for i in range(1, n_steps):
        theta = curl * i
        bend = perp * np.sin(theta) * step_len
        step = direction * step_len + bend
        positions.append(positions[-1] + step)
    positions = np.array(positions)

    times = t0 + np.arange(n_steps) * (step_len / 200.0)
    dep = np.full(n_steps, dedx) * rng.uniform(0.7, 1.3, n_steps)

    return positions, times, dep


vertex_pos = np.array([0.0, 0.0, 0.0])
t0 = 0.0

particles = [
    (11, np.array([1, 0.3, 0.2]), 40, 15.0, t0, 0.3, 0.15),  # electron, curls in field
    (2212, np.array([-0.5, 1, 0.1]), 25, 20.0, t0 + 0.5, 1.2, 0.0),  # proton, minimal curl
    (22, np.array([-0.2, -0.8, 0.9]), 30, 25.0, t0 + 0.2, 0.05, 0.0),  # gamma, light dep.
]

pdg = []
pos_x, pos_y, pos_z, time, deposited_energy = [], [], [], [], []
for pdg_code, direction, n_steps, step_len, start_t, dedx, curl in particles:
    positions, times, dep = make_track(vertex_pos, direction, n_steps, step_len, start_t, dedx, curl)
    pdg.append(pdg_code)
    pos_x.append(positions[:, 0].tolist())
    pos_y.append(positions[:, 1].tolist())
    pos_z.append(positions[:, 2].tolist())
    time.append(times.tolist())
    deposited_energy.append(dep.tolist())

synthetic_tracks = {
    "pdg": np.array(pdg),
    "pos_x": ak.Array(pos_x),
    "pos_y": ak.Array(pos_y),
    "pos_z": ak.Array(pos_z),
    "time": ak.Array(time),
    "deposited_energy": ak.Array(deposited_energy),
}
synthetic_tracks

## Build synthetic truth vertices

`add_vertices` expects flat arrays `pdg`, `x`, `y`, `z`, `t`, `energy`, one entry per vertex.

In [ ]:
vertex_2 = vertex_pos + np.array([5, -3, 2])

synthetic_vertices = {
    "pdg": np.array([11, 2212]),
    "x": np.array([vertex_pos[0], vertex_2[0]]),
    "y": np.array([vertex_pos[1], vertex_2[1]]),
    "z": np.array([vertex_pos[2], vertex_2[2]]),
    "t": np.array([0.0, 1.1]),
    "energy": np.array([2.5, 0.8]),
}
synthetic_vertices

## Assemble the synthetic display

`add_deposits` bins every step's deposited energy into 10 mm voxels, giving a calorimetric view alongside the track lines and vertex markers. Pass the same `t_min`/`t_max` to tracks and vertices so they share one time colour bar.

In [ ]:
all_times = ak.to_numpy(ak.flatten(synthetic_tracks["time"]))
t_min, t_max = float(all_times.min()), float(all_times.max())

(
    EventDisplay(title="Synthetic event: e- / p / gamma from a common vertex", height=650)
    .add_deposits(
        synthetic_tracks["pos_x"],
        synthetic_tracks["pos_y"],
        synthetic_tracks["pos_z"],
        synthetic_tracks["deposited_energy"],
        voxel_size=10.0,
    )
    .add_tracks(**synthetic_tracks, t_min=t_min, t_max=t_max)
    .add_vertices(**synthetic_vertices, t_min=t_min, t_max=t_max)
)

## Real data: reading an event from a ROOT ntuple

`load_event` opens a SNO+ `imaging_reco` ntuple with `uproot` and returns `(tracks, vertices)` kwargs dicts, ready to unpack straight into `add_tracks`/`add_deposits`/`add_vertices` with no manual column wrangling.

This reads event 1 from a 2.5 MeV uniform-fiducial electron MC sample, which has a handful of bremsstrahlung/ionization secondaries in addition to the primary electron.

In [ ]:
path = "/data/snoplus2/degraw/imaging_reco/mc/electron_uniform_fiducial/root/job_0.ntuple.root"
real_tracks, real_vertices = load_event(path, index=1)

print(f"{len(real_tracks['pdg'])} tracks, pdgs={ak.to_list(real_tracks['pdg'])}")

In [ ]:
real_times = ak.to_numpy(ak.flatten(real_tracks["time"]))
t_min, t_max = float(real_times.min()), float(real_times.max())

(
    EventDisplay(title="Real event: 2.5 MeV electron, uniform fiducial MC", height=650)
    .add_deposits(
        real_tracks["pos_x"],
        real_tracks["pos_y"],
        real_tracks["pos_z"],
        real_tracks["deposited_energy"],
        voxel_size=50.0,
    )
    .add_tracks(**real_tracks, t_min=t_min, t_max=t_max)
    .add_vertices(**real_vertices, t_min=t_min, t_max=t_max)
)

## Reconstructed voxel images

`add_volume` takes a reconstructed image, either as a dense `(nx, ny, nz)` grid with the `origin` of its min corner or as sparse voxel `centers`/`energies`. Extra polylines (for example a fitted axis) go in with `add_line`. Here a blurred straight track stands in for an MLEM image.

In [ ]:
voxel = 50.0
axis = np.array([[-250.0, -100.0, -200.0], [250.0, 150.0, 250.0]])
g = np.arange(-400, 400 + voxel, voxel)
X, Y, Z = np.meshgrid(g, g, g, indexing="ij")
pts = np.stack([X, Y, Z], axis=-1)
d = axis[1] - axis[0]
s = np.clip(((pts - axis[0]) @ d) / (d @ d), 0, 1)
dist = np.linalg.norm(pts - (axis[0] + s[..., None] * d), axis=-1)
image = np.exp(-0.5 * (dist / 60.0) ** 2) * (0.6 + 0.4 * s)

(
    EventDisplay(title="Toy reconstructed image, 50 mm voxels")
    .add_volume(image, origin=g[0] - voxel / 2, voxel_size=voxel, name="image", colorbar_title="Intensity")
    .add_line(axis, "true track", color="#00a0ff")
)